# Residuales vs $Y$ observado: por qué engaña (demo interactiva)

Aquí se ve con simulación que, incluso con errores homocedásticos,
el scatter **residuales vs $Y$ observado** puede mostrar un patrón artificial.

**Modelo:**  
$$Y = \beta_0 + \beta_1 X + \varepsilon,\quad \varepsilon\sim \mathcal N(0,\sigma^2)$$

Identidad clave:  
$$\hat\varepsilon = Y-\hat Y\ \Rightarrow\ Y=\hat Y+\hat\varepsilon$$

Si graficas $\hat\varepsilon$ vs $Y$, estás metiendo el residual **en ambos ejes**.  


In [1]:
# (Colab) habilitar widgets
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception as e:
    print("Si no estás en Colab, ignora esto:", e)

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, Checkbox


## Interactivo

- **Caso homocedástico**: varianza constante (debería “verse bien”).  
- **Caso heterocedástico**: varianza crece con $|X|$ (aparece abanico).  

Compara:
- $\hat\varepsilon$ vs $Y$ (observado)  ❌
- $\hat\varepsilon$ vs $\hat Y$ (ajustado) ✔


In [2]:
# @title
def run_demo(n=250, sigma=1.0, beta1=2.0, hetero=False, hetero_strength=0.8, seed=42):
    rng = np.random.default_rng(seed)
    x = rng.uniform(-2, 2, size=n)
    beta0 = 1.0

    if hetero:
        # sigma(x) = sigma * (1 + a*|x|)
        sig_i = sigma * (1.0 + hetero_strength*np.abs(x))
        eps = rng.normal(0.0, sig_i, size=n)
    else:
        eps = rng.normal(0.0, sigma, size=n)

    y = beta0 + beta1*x + eps

    # OLS simple
    X = np.column_stack([np.ones(n), x])
    bhat = np.linalg.inv(X.T@X) @ (X.T@y)
    yhat = X @ bhat
    resid = y - yhat

    corr_resid_y = float(np.corrcoef(resid, y)[0,1])
    corr_resid_yhat = float(np.corrcoef(resid, yhat)[0,1])

    # Plot 1: resid vs y (observado) — puede engañar
    plt.figure()
    plt.scatter(y, resid, s=14, alpha=0.7)
    plt.axhline(0, linewidth=1)
    plt.title(rf"Residuales vs $Y$ observado   corr={corr_resid_y:.3f}")
    plt.xlabel(r"$Y$ observado")
    plt.ylabel("Residual")
    plt.show()

    # Plot 2: resid vs yhat (ajustado) — diagnóstico estándar
    plt.figure()
    plt.scatter(yhat, resid, s=14, alpha=0.7)
    plt.axhline(0, linewidth=1)
    plt.title(rf"Residuales vs $\hat{{Y}}$ (ajustado)   corr={corr_resid_yhat:.3f}")
    plt.xlabel(r"$\hat{Y}$")
    plt.ylabel("Residual")
    plt.show()

    print(f"b0_hat={bhat[0]:.3f}, b1_hat={bhat[1]:.3f}")
    print(f"corr(resid, Y observado) = {corr_resid_y:.3f}")
    print(f"corr(resid, Y ajustado)  = {corr_resid_yhat:.3f}")


In [3]:
interact(
    run_demo,
    n=IntSlider(250, min=50, max=2000, step=50, description="n"),
    sigma=FloatSlider(1.0, min=0.2, max=5.0, step=0.1, description="sigma"),
    beta1=FloatSlider(2.0, min=-4.0, max=4.0, step=0.2, description="beta1"),
    hetero=Checkbox(False, description="heterocedástico"),
    hetero_strength=FloatSlider(0.8, min=0.0, max=2.0, step=0.1, description="a"),
    seed=IntSlider(42, min=0, max=999, step=1, description="seed"),
);


interactive(children=(IntSlider(value=250, description='n', max=2000, min=50, step=50), FloatSlider(value=1.0,…

### Conclusión (una línea)

- Para homocedasticidad: **usa** $\hat\varepsilon$ vs $\hat Y$ (o vs $X$ en simple).  
- Evita $\hat\varepsilon$ vs $Y$ observado: introduce **dependencia mecánica**.  
